In [92]:
import pandas as pd

import psycopg2

class PsqlDatabaseManager:

    def __init__(self, database, host, user, password, port):

        self.database = database
        self.host = host
        self.user = user
        self.password = password
        self.port = port
        self.conn = None
        self.cursor = None
        self.connect()

    def connect(self):
        try:
            self.conn = psycopg2.connect(database=self.database,
                                         host=self.host,
                                         user=self.user,
                                         password=self.password,
                                         port=self.port)
            self.cursor = self.conn.cursor()
        except psycopg2.DatabaseError as e:
            raise Exception(f"Error database connection: {e}")

    def list_tables(self):
        query = "SELECT table_name FROM information_schema.tables WHERE table_schema = 'public'"
        results = self.fetch_results(query)
        return results        
    
    def create_table(self, table_name, columns):
        query = f"CREATE TABLE {table_name} ({columns})"
        self.execute_query(query)

    def drop_table(self, table_name):
        query = f"DROP TABLE {table_name}"
        self.execute_query(query)

    def insert_data(self, table_name, data):
        query = f"INSERT INTO {table_name} VALUES {data}"
        self.execute_query(query)

    def insert_data_from_df(self, table_name, df):
        columns = ", ".join(df.columns)
        data = ", ".join([str(tuple(row)) for row in df.values])
        query = f"INSERT INTO {table_name} ({columns}) VALUES {data}"
        self.execute_query(query)            

    def execute_query(self, query):
        try:
            self.cursor.execute(query)
            self.conn.commit()
        except psycopg2.DatabaseError as e:
            self.conn.rollback()
            raise Exception(f"Error executing query: {e}")

    def fetch_results(self, query):
        try:
            self.cursor.execute(query)
            results = self.cursor.fetchall()
            return results
        except psycopg2.DatabaseError as e:
            raise Exception(f"Error fetching results: {e}")

    def close(self):
        if self.cursor:
            self.cursor.close()
        if self.conn:
            self.conn.close()        

    def is_db_empty(self):
        query = "SELECT * FROM information_schema.tables WHERE table_schema = 'public'"
        results = self.fetch_results(query)
        return len(results) == 0        

In [94]:
# about user
USER_DB = 'adrie'
PASSWORD_DB = '16871687'
HOST_DB = 'localhost'
PORT_DB = '5432'
# about db
NAME_DB = 'test_db'

# Database connection parameters
db_params = {
    'user': USER_DB,
    'password': PASSWORD_DB,
    'host': HOST_DB,
    'port': PORT_DB,
    'database': NAME_DB,
}

db_manager = PsqlDatabaseManager(**db_params)

In [96]:
# create table

table_name = 'rofl_table'
columns = 'id SERIAL PRIMARY KEY, name VARCHAR(255), age INT'
db_manager.create_table(table_name, columns)



